# ACCIDENT dataset — step 0: download and inspect
Run all cells (Runtime > Run all). It downloads the dataset from Kaggle, prints its structure, and saves `inspect_report.txt`. Send that file (or paste its content) back; training comes next. Needs a Kaggle account: kaggle.com > Settings > API > *Create New Token* (gives `kaggle.json`), and accept the dataset's terms once on https://www.kaggle.com/datasets/picekl/accident

In [ ]:
!pip -q install kaggle opencv-python-headless pandas
import subprocess; print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout or 'no GPU')

In [ ]:
import os, json
from google.colab import files
print('Upload kaggle.json')
up = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d picekl/accident -p /content/accident --unzip -q
!du -sh /content/accident

In [ ]:
import pathlib, collections, pandas as pd, cv2
R = pathlib.Path('/content/accident'); out = []
def p(*a):
    s = ' '.join(str(x) for x in a); print(s); out.append(s)
p('== top-level tree (3 levels) ==')
for path in sorted(R.rglob('*')):
    d = len(path.relative_to(R).parts)
    if d <= 3 and (path.is_dir() or d <= 2): p('  ' * (d-1) + path.name + ('/' if path.is_dir() else ''))
exts = collections.Counter(x.suffix.lower() for x in R.rglob('*') if x.is_file())
p('== file types ==', dict(exts))
for csv in sorted(R.rglob('*.csv')):
    df = pd.read_csv(csv)
    p(f'== {csv.relative_to(R)}  shape={df.shape}')
    p('columns:', list(df.columns)); p(df.head(5).to_string())
    for c in df.columns:
        if df[c].nunique() <= 12: p(f'  value counts {c}:', df[c].value_counts().to_dict())
for t in list(R.rglob('*.md')) + list(R.rglob('LICENSE*')) + list(R.rglob('*.txt')):
    p('== text file', t.relative_to(R)); p(t.read_text(errors='ignore')[:1500])
vids = sorted([x for x in R.rglob('*') if x.suffix.lower() in ('.mp4','.avi','.mov','.mkv','.webm')])
p('== videos:', len(vids))
for v in vids[:5] + vids[-2:]:
    c = cv2.VideoCapture(str(v)); p(v.relative_to(R), 'fps', c.get(5), 'frames', int(c.get(7)), 'size', int(c.get(3)), 'x', int(c.get(4)))
open('inspect_report.txt','w').write('\n'.join(out))
files.download('inspect_report.txt')